# Notebook 06 — Công cụ tự chú thích hình trong tài liệu Word (1A)

Tải lên một file `.docx` → công cụ tìm từng hình → sinh chú thích bằng LLaVA-OneVision-7B → chèn **"Hình n. ..."** ngay dưới hình → thêm **Danh mục hình** ở cuối → tải file kết quả về và mở bằng Microsoft Word.

- Chú thích dùng kiểu **Caption** và **trường SEQ** của Word, giống References → Insert Caption: thêm/xoá hình thì số tự cập nhật.
- Đoạn văn trước và sau hình được đưa vào chỉ dẫn làm ngữ cảnh (cải tiến 2A).
- Tùy chọn bật bước tự kiểm tra (cải tiến 2B), chậm hơn khoảng 5 lần.
- Chú thích gõ tay sẵn có ("Hình 5. ...") được giữ nguyên phần chữ, chỉ đổi số thành tự động.
- Giới hạn hiện tại: chỉ xử lý ảnh chèn trực tiếp trong đoạn văn, chưa xử lý ảnh nằm trong bảng hoặc khung văn bản.

**Cần GPU:** Runtime → Change runtime type → T4 GPU.

## 1. Cấu hình

In [ ]:
CAPTION_LANG = "vi"     # "vi": chú thích tiếng Việt; "en": tiếng Anh
USE_CONTEXT = True      # cải tiến 2A: dùng đoạn văn quanh hình
SELF_CHECK = False      # cải tiến 2B: tự kiểm tra, chậm hơn
LABEL = "Hình"          # nhãn đánh số, ví dụ "Hình" hoặc "Figure"

## 2. Cài đặt và nạp mô hình

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes python-docx

In [ ]:
import copy
import re

from docx.enum.style import WD_STYLE_TYPE
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Pt

CAPTION_RE = re.compile(r"^\s*(Hình|Figure|Fig\.?)\s*\d", re.IGNORECASE)
CHAPTER_RE = re.compile(r"^\s*(?:Hình|Figure|Fig\.?)\s*\d+[.\-]\d+", re.IGNORECASE)
NUMBERED_RE = re.compile(r"^\s*(?:Hình|Figure|Fig\.?)\s*\d+\s*[.:]?\s*(.*)$", re.IGNORECASE | re.DOTALL)
BLIP = qn("a:blip")
EMBED = qn("r:embed")


def image_blobs(paragraph):
    """Ảnh nằm trực tiếp trong đoạn văn (ảnh chèn inline, kiểu phổ biến nhất trong Word)."""
    part = paragraph.part
    blobs = []
    for blip in paragraph._p.iter(BLIP):
        rel_id = blip.get(EMBED)
        if rel_id and rel_id in part.related_parts:
            blobs.append(part.related_parts[rel_id].blob)
    return blobs


def is_caption(paragraph):
    return paragraph is not None and (paragraph.style.name.lower() == "caption" or bool(CAPTION_RE.match(paragraph.text)))


def surrounding_text(paragraphs, index, window=2, max_chars=1200):
    """Đoạn văn trước và sau hình, dùng làm ngữ cảnh cho mô hình (ý tưởng 2A)."""
    picked = []
    for j in list(range(index - window, index)) + list(range(index + 1, index + 1 + window)):
        if 0 <= j < len(paragraphs):
            text = paragraphs[j].text.strip()
            if text and not is_caption(paragraphs[j]):
                picked.append(text)
    return " ".join(picked)[:max_chars]


def ensure_caption_style(doc):
    try:
        return doc.styles["Caption"]
    except KeyError:
        style = doc.styles.add_style("Caption", WD_STYLE_TYPE.PARAGRAPH)
        style.base_style = doc.styles["Normal"]
        style.font.italic = True
        style.font.size = Pt(10)
        return style


def simple_field(instr, shown):
    field = OxmlElement("w:fldSimple")
    field.set(qn("w:instr"), instr)
    run = OxmlElement("w:r")
    text = OxmlElement("w:t")
    text.text = shown
    run.append(text)
    field.append(run)
    return field


def insert_caption_after(paragraph, number, caption, label="Hình"):
    """Chèn chú thích giống References → Insert Caption: kiểu Caption + trường SEQ tự đánh số."""
    new_p = OxmlElement("w:p")
    paragraph._p.addnext(new_p)
    from docx.text.paragraph import Paragraph
    cap = Paragraph(new_p, paragraph._parent)
    cap.style = "Caption"
    cap.add_run(f"{label} ")
    cap._p.append(simple_field(f" SEQ {label} \\* ARABIC ", str(number)))
    cap.add_run(f". {caption}")
    return cap


def add_table_of_figures(doc, label="Hình", title="Danh mục hình"):
    """Thêm mục lục hình ở cuối tài liệu; Word cập nhật khi mở file (hoặc nhấn F9)."""
    doc.add_heading(title, level=1)
    p = doc.add_paragraph()
    p._p.append(simple_field(f' TOC \\h \\z \\c "{label}" ', "Mở bằng Word và nhấn F9 để cập nhật danh mục hình."))
    settings = doc.settings.element
    if settings.find(qn("w:updateFields")) is None:
        flag = OxmlElement("w:updateFields")
        flag.set(qn("w:val"), "true")
        # Lược đồ của Word quy định thứ tự phần tử: updateFields phải đứng trước các phần tử này
        later = {qn("w:" + n) for n in (
            "hdrShapeDefaults", "footnotePr", "endnotePr", "compat", "docVars", "rsids", "mathPr",
            "attachedSchema", "themeFontLang", "clrSchemeMapping", "doNotIncludeSubdocsInStats",
            "doNotAutoCompressPictures", "forceUpgrade", "captions", "readModeInkLockDown", "smartTagType",
            "schemaLibrary", "shapeDefaults", "doNotEmbedSmartTags", "decimalSymbol", "listSeparator")}
        anchor = next((child for child in settings if child.tag in later), None)
        if anchor is None:
            settings.append(flag)
        else:
            anchor.addprevious(flag)


def caption_document(doc, caption_fn, label="Hình", add_toc=True):
    """caption_fn(image_bytes, context) -> câu chú thích. Trả về danh sách hình đã xử lý."""
    ensure_caption_style(doc)
    paragraphs = list(doc.paragraphs)
    report, number = [], 0
    for i, paragraph in enumerate(paragraphs):
        blobs = image_blobs(paragraph)
        if not blobs:
            continue
        nxt = paragraphs[i + 1] if i + 1 < len(paragraphs) else None
        number += 1
        if is_caption(nxt):
            if CHAPTER_RE.match(nxt.text):
                # Đánh số theo chương ("Hình 3.1"): giữ nguyên, công cụ chưa hỗ trợ kiểu này
                report.append({"number": number, "status": "đánh số theo chương, giữ nguyên", "caption": nxt.text})
            elif "SEQ" in nxt._p.xml:
                report.append({"number": number, "status": "đã có chú thích tự đánh số, giữ nguyên", "caption": nxt.text})
            else:
                # Chú thích gõ tay: giữ phần chữ, đổi số thành trường SEQ để đánh số thống nhất
                text = NUMBERED_RE.match(nxt.text).group(1).strip()
                insert_caption_after(paragraph, number, text.rstrip(".") + ".", label)
                nxt._p.getparent().remove(nxt._p)
                report.append({"number": number, "status": "đã có chú thích gõ tay, đổi số thành tự động", "caption": text})
            continue
        context = surrounding_text(paragraphs, i)
        caption = caption_fn(blobs[0], context).strip().rstrip(".") + "."
        insert_caption_after(paragraph, number, caption, label)
        report.append({"number": number, "status": "đã thêm", "caption": caption, "context": context[:200]})
    if add_toc and number:
        add_table_of_figures(doc, label)
    return report

In [ ]:
import io, re
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16))


def chat(text, image=None, max_new_tokens=80):
    content = ([{"type": "image", "image": image}] if image is not None else []) + [{"type": "text", "text": text}]
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt")
    inputs = inputs.to(model.device, torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

## 3. Hàm sinh chú thích (ngữ cảnh + tự kiểm tra)

In [ ]:
LANG_NAME = {"vi": "Vietnamese", "en": "English"}[CAPTION_LANG]


def self_check(caption, image):
    claims_text = chat("Split the following caption into short, separate factual statements, one per line. "
                       "Include every object, attribute, number and trend it mentions. Caption: " + caption,
                       max_new_tokens=120)
    claims = [re.sub(r"^[\s\-\*\d\.\)]+", "", line).strip() for line in claims_text.splitlines()]
    claims = [c for c in claims if len(c.split()) >= 2][:6]
    wrong = [c for c in claims
             if not chat(f"Is the following statement true for this image? Answer yes or no.\nStatement: {c}",
                         image, max_new_tokens=3).lower().startswith("yes")]
    if not wrong:
        return caption
    return chat(f"Rewrite this caption as one short sentence in {LANG_NAME}, removing the incorrect statements.\n"
                f"Caption: {caption}\nIncorrect statements: " + "; ".join(wrong), image)


def caption_fn(image_bytes, context):
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    if USE_CONTEXT and context:
        prompt = (f"Text around this figure in the document:\n\"{context}\"\n"
                  f"Write a caption for the figure in one short sentence in {LANG_NAME}.")
    else:
        prompt = f"Write a caption for this figure in one short sentence in {LANG_NAME}."
    caption = chat(prompt, image)
    if SELF_CHECK:
        caption = self_check(caption, image)
    print(f"  → {caption}")
    return caption

## 4. Tải file Word lên, xử lý và tải kết quả về

In [ ]:
from google.colab import files
from docx import Document

uploaded = files.upload()
for name, data in uploaded.items():
    if not name.lower().endswith(".docx"):
        print("Bỏ qua (không phải .docx):", name)
        continue
    print("Đang xử lý:", name)
    doc = Document(io.BytesIO(data))
    report = caption_document(doc, caption_fn, label=LABEL)
    out_name = name[:-5] + "_chu_thich.docx"
    doc.save(out_name)
    for r in report:
        print(f"{LABEL} {r['number']}: {r['status']} | {r['caption']}")
    files.download(out_name)
print("Mở file kết quả bằng Word; nếu Word hỏi cập nhật trường (fields), chọn Yes để danh mục hình hiện đầy đủ.")